# Behavioural Risk Stratification (Engine 1) + PHQ-9 Validation (Engine 2)

## Leading introduction: how the two engines are used together

This notebook implements a dual-engine decision workflow for intervention support.

Engine 1 is the primary behavioural decision engine. It transforms phone-use behaviours into interpretable domain scores, behavioural risk tiers, and top behavioural drivers. These outputs are designed for direct action planning because they identify modifiable patterns.

Engine 2 is a secondary association engine. It estimates the probability of PHQ9_10plus from behavioural signals to provide supporting context about mental-health association risk. Engine 2 does not trigger intervention by itself; it is used to calibrate confidence and urgency around the Engine 1 recommendation.

In operational use, the workflow is:
1. Run Engine 1 to produce risk tier and behavioural explanation.
2. Run Engine 2 to add association context for prioritization review.
3. Make decisions with Engine 1 as the trigger and Engine 2 as supporting evidence.

## Objective and boundary

### Engine 1 (Primary intervention trigger)
Input: phone-use behaviour features only  
Output: behavioural risk tier + domain scores + top behavioural drivers  
Purpose: identify modifiable behavioural patterns for intervention support  
Method: unsupervised or weakly supervised scoring framework

### Engine 2 (Secondary validation only)
Input: phone-use behaviour features (plus optional app indicators)  
Output: PHQ9_10plus risk probability (`phq9_association_risk_estimate`)  
Purpose: test association between behavioural risk signals and elevated depressive symptom burden  
Method: supervised model

**Operational rule:** Engine 1 drives intervention workflow. Engine 2 is supporting evidence and does not replace the primary trigger.

## Notebook structure
1. Step 1: Setup + load data + quick diagnostics
2. Step 2: Fail-fast schema and range validation
3. Step 3: Behavioural domain engineering
4. Step 4: Theory-driven and data-driven behavioural representation
5. Step 5: Engine 1 risk tier and top drivers
6. Step 6: Engine 2 model comparison (CV)
7. Step 7: Threshold and confusion-matrix trade-off
8. Step 7B: Bootstrap confidence intervals for uncertainty quantification
9. Step 8: Final signal-to-decision table
10. Step 9: Example users (3 contrasting users)

## Step 1 - Setup, load canonical data, and quick diagnostics

### Methodology
1. Create a reproducible environment by importing required libraries and fixing a random seed.
2. Load the canonical cleaned dataset from the project data folder.
3. Run quick quality checks: table size, target class distribution, and missing target values.

### Why this matters in this step
1. It confirms that data is readable and usable before any modeling work starts.
2. It highlights class imbalance early so later model evaluation can be designed correctly.

### Why this matters for the full analysis
1. Reproducibility makes results auditable and easier to compare across reruns.
2. Early diagnostics prevent hidden data issues from contaminating all later conclusions.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import precision_score, f1_score, recall_score, balanced_accuracy_score, confusion_matrix

SEED = 42
np.random.seed(SEED)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

df = pd.read_csv('../data/DSA4262_Survey_Cleaned.csv')
print('Shape:', df.shape)
print('\nPHQ9_10plus distribution:')
print(df['PHQ9_10plus'].value_counts(dropna=False))
print('\nMissing PHQ9_10plus:', int(df['PHQ9_10plus'].isna().sum()))

Shape: (50, 97)

PHQ9_10plus distribution:
PHQ9_10plus
0    38
1    12
Name: count, dtype: int64

Missing PHQ9_10plus: 0


**Step 1 evaluation**
- What the code did: Loaded the canonical dataset and printed dataset shape, target-class counts, and target missingness.
- What result was obtained: Data shape is `50 x 97`; `PHQ9_10plus` has `38` negatives and `12` positives; missing target count is `0`.
- Why this matters for decisions: The positive class is minority, so Engine 2 must be evaluated with imbalance-aware metrics and threshold analysis.

## Step 2 - Fail-fast schema and range validation

### Methodology
1. Check that all required columns are present.
2. Validate that each survey variable stays within its expected numeric scale.
3. Standardize the target (`PHQ9_10plus`) into valid binary form and verify values are only 0/1.

### Why this matters in this step
1. It catches structural and value errors immediately, before feature engineering.
2. It guarantees that scale meaning is preserved for later domain scoring.

### Why this matters for the full analysis
1. Valid input data is the foundation of trustworthy model outputs.
2. Without this gate, downstream metrics and interpretations can look correct but be misleading.

In [2]:
required_cols = ['PHQ9_Total', 'PHQ9_10plus'] + [
    'Q4_DailyPhoneTime_num', 'Q5_CheckFrequency_num', 'Q6_AfterWakeCheck_num',
    'Q7_After11PMUse_num', 'Q8_StopUseAtNight_num', 'Q10_SocialMediaTime_num',
    'Q11_PhoneDuringMeals_num', 'Q12_PhoneDuringWorkClass_num', 'Q13_CheckDuringTasks_num',
    'Q14_NotificationsInterrupt_num', 'Q15_UseWhenBored_num', 'Q16_UseWhenStressed_num',
    'Q17_DistractNegativeEmotions_num', 'Q18_PhoneInBed_num', 'Q19_PhoneDelaysSleep_num',
    'Q20_NotificationsWakeNight_num', 'Q21_WakeToCheckPhone_num', 'Q22_UnlockWithoutReason_num',
    'Q23_UseLongerThanIntended_A_num', 'Q24_FailToReduceUse_num',
    'Q25_MissPlannedWork_num', 'Q26_ConcentrationProblems_num', 'Q27_PhysicalPain_num',
    'Q28_CannotStandWithoutPhone_num', 'Q29_ImpatientWithoutPhone_num', 'Q30_PhoneOnMyMind_num',
    'Q31_WontGiveUpPhone_num', 'Q32_ConstantChecking_num', 'Q33_UseLongerThanIntended_B_num',
    'Q34_OthersSayTooMuchUse_num',
    'Q35_PHQ1_Interest_num', 'Q36_PHQ2_Down_num', 'Q37_PHQ3_Sleep_num', 'Q38_PHQ4_Tired_num',
    'Q39_PHQ5_Appetite_num', 'Q40_PHQ6_SelfWorth_num', 'Q41_PHQ7_Concentration_num',
    'Q42_PHQ8_RestlessSlow_num', 'Q43_PHQ9_SelfHarm_num',
]
missing_cols = [c for c in required_cols if c not in df.columns]
assert not missing_cols, f'Missing required columns: {missing_cols}'

range_rules = {
    'Q4_DailyPhoneTime_num': (1, 5), 'Q5_CheckFrequency_num': (1, 4), 'Q6_AfterWakeCheck_num': (1, 4),
    'Q7_After11PMUse_num': (1, 5), 'Q8_StopUseAtNight_num': (1, 5), 'Q10_SocialMediaTime_num': (1, 5),
    'PHQ9_Total': (0, 27),
}
for c in [
    'Q11_PhoneDuringMeals_num','Q12_PhoneDuringWorkClass_num','Q13_CheckDuringTasks_num','Q14_NotificationsInterrupt_num',
    'Q15_UseWhenBored_num','Q16_UseWhenStressed_num','Q17_DistractNegativeEmotions_num','Q18_PhoneInBed_num',
    'Q19_PhoneDelaysSleep_num','Q20_NotificationsWakeNight_num','Q21_WakeToCheckPhone_num','Q22_UnlockWithoutReason_num',
    'Q23_UseLongerThanIntended_A_num','Q24_FailToReduceUse_num',
]:
    range_rules[c] = (1, 5)
for c in [
    'Q25_MissPlannedWork_num','Q26_ConcentrationProblems_num','Q27_PhysicalPain_num','Q28_CannotStandWithoutPhone_num',
    'Q29_ImpatientWithoutPhone_num','Q30_PhoneOnMyMind_num','Q31_WontGiveUpPhone_num','Q32_ConstantChecking_num',
    'Q33_UseLongerThanIntended_B_num','Q34_OthersSayTooMuchUse_num',
]:
    range_rules[c] = (1, 6)
for c in [
    'Q35_PHQ1_Interest_num','Q36_PHQ2_Down_num','Q37_PHQ3_Sleep_num','Q38_PHQ4_Tired_num',
    'Q39_PHQ5_Appetite_num','Q40_PHQ6_SelfWorth_num','Q41_PHQ7_Concentration_num',
    'Q42_PHQ8_RestlessSlow_num','Q43_PHQ9_SelfHarm_num',
]:
    range_rules[c] = (0, 3)

for col, (lo, hi) in range_rules.items():
    s = pd.to_numeric(df[col], errors='coerce')
    out = s.dropna().loc[(s.dropna() < lo) | (s.dropna() > hi)]
    assert out.empty, f'{col} out of range [{lo}, {hi}] count={len(out)}'

if df['PHQ9_10plus'].dtype == bool:
    df['PHQ9_10plus'] = df['PHQ9_10plus'].astype(int)
elif str(df['PHQ9_10plus'].dtype) == 'object':
    df['PHQ9_10plus'] = df['PHQ9_10plus'].map({True: 1, False: 0, 'True': 1, 'False': 0, '1': 1, '0': 0})
valid_vals = set(df['PHQ9_10plus'].dropna().astype(int).unique().tolist())
assert valid_vals.issubset({0, 1}), f'Invalid PHQ9_10plus values: {valid_vals}'

print('Schema checks passed: required columns, ranges, and target encoding are valid.')

Schema checks passed: required columns, ranges, and target encoding are valid.


**Step 2 evaluation**
- What the code did: Ran fail-fast checks for required columns, item value ranges, and binary target validity.
- What result was obtained: All schema and range assertions passed; `PHQ9_10plus` is valid binary input.
- Why this matters for decisions: Engine outputs are trustworthy only when measurement and encoding assumptions are satisfied before modeling.

## Step 3 - Behavioural domain engineering

### Methodology
1. Convert raw item responses from different Likert ranges to a common 0-100 scale.
2. Group related items into behavioural domains (exposure, context, emotional, sleep, self-regulation).
3. Compute an optional PSU-style severity score from Q25-Q34 items.

### Why this matters in this step
1. Normalization makes different question scales directly comparable.
2. Domain aggregation reduces noise and keeps outputs easier to explain.

### Why this matters for the full analysis
1. These domain features are the core inputs for Engine 1 risk stratification.
2. They also provide a compact, interpretable feature space for improved Engine 2 modeling.

In [3]:
domain_map = {
    'exposure_intensity': [
        'Q4_DailyPhoneTime_num','Q5_CheckFrequency_num','Q6_AfterWakeCheck_num',
        'Q7_After11PMUse_num','Q8_StopUseAtNight_num','Q10_SocialMediaTime_num'
    ],
    'context_disruption': [
        'Q11_PhoneDuringMeals_num','Q12_PhoneDuringWorkClass_num','Q13_CheckDuringTasks_num','Q14_NotificationsInterrupt_num'
    ],
    'emotional_coping': [
        'Q15_UseWhenBored_num','Q16_UseWhenStressed_num','Q17_DistractNegativeEmotions_num'
    ],
    'sleep_disruption': [
        'Q18_PhoneInBed_num','Q19_PhoneDelaysSleep_num','Q20_NotificationsWakeNight_num','Q21_WakeToCheckPhone_num'
    ],
    'self_regulation': [
        'Q22_UnlockWithoutReason_num','Q23_UseLongerThanIntended_A_num','Q24_FailToReduceUse_num'
    ],
}
psu_cols = [
    'Q25_MissPlannedWork_num','Q26_ConcentrationProblems_num','Q27_PhysicalPain_num','Q28_CannotStandWithoutPhone_num',
    'Q29_ImpatientWithoutPhone_num','Q30_PhoneOnMyMind_num','Q31_WontGiveUpPhone_num','Q32_ConstantChecking_num',
    'Q33_UseLongerThanIntended_B_num','Q34_OthersSayTooMuchUse_num'
]

def norm_0_100(series, lo, hi):
    return (series - lo) / (hi - lo) * 100

item_range = {}
for c in domain_map['exposure_intensity']:
    item_range[c] = (1, 4) if c in {'Q5_CheckFrequency_num', 'Q6_AfterWakeCheck_num'} else (1, 5)
for k in ['context_disruption', 'emotional_coping', 'sleep_disruption', 'self_regulation']:
    for c in domain_map[k]:
        item_range[c] = (1, 5)
for c in psu_cols:
    item_range[c] = (1, 6)

for dname, cols in domain_map.items():
    scaled = [norm_0_100(df[c].astype(float), *item_range[c]) for c in cols]
    df[f'domain_{dname}'] = pd.concat(scaled, axis=1).mean(axis=1)
df['psu_style_score'] = pd.concat([norm_0_100(df[c].astype(float), 1, 6) for c in psu_cols], axis=1).mean(axis=1)

domain_cols = [c for c in df.columns if c.startswith('domain_')]
display(df[domain_cols + ['psu_style_score']].describe().T[['mean', 'std', 'min', 'max']])

,mean,std,min,max
domain_exposure_intensity,69.527778,15.246593,37.50,94.444444
domain_context_disruption,60.750000,14.673313,37.50,87.500000
domain_emotional_coping,71.833333,20.612091,0.00,100.000000
domain_sleep_disruption,44.500000,11.268409,6.25,62.500000
domain_self_regulation,56.000000,20.135595,0.00,100.000000
psu_style_score,43.480000,17.837406,14.00,90.000000


**Step 3 evaluation**
- What the code did: Normalized mixed Likert items and aggregated them into behavioural domain scores plus PSU-style severity.
- What result was obtained: Domain means are exposure `69.53`, context `60.75`, emotional `71.83`, sleep `44.50`, self-regulation `56.00`, PSU-style `43.48`.
- Why this matters for decisions: Domain scores provide interpretable behavioural dimensions that support transparent Engine 1 tiering and messaging.

## Step 4 - Theory-driven and data-driven behavioural representation

### Methodology
1. Create theory-driven interaction features (for example exposure x sleep).
2. Apply PCA to summarize high-dimensional behaviour items into principal components.
3. Use KMeans clustering to group users with similar behaviour patterns.

### Why this matters in this step
1. Interaction terms capture joint behaviour effects that single variables miss.
2. PCA and clustering reveal hidden structure and profile diversity in the data.

### Why this matters for the full analysis
1. This improves understanding of behaviour patterns behind risk scores.
2. It supports interpretability and quality checks before final decision outputs are produced.

In [4]:
df['domain_exposure_sleep_interaction'] = (df['domain_exposure_intensity'] * df['domain_sleep_disruption']) / 100
df['domain_emotion_selfreg_interaction'] = (df['domain_emotional_coping'] * df['domain_self_regulation']) / 100

behaviour_item_cols = sorted({c for cols in domain_map.values() for c in cols}.union(set(psu_cols)))
X_items = df[behaviour_item_cols].astype(float).fillna(df[behaviour_item_cols].median())

X_std = StandardScaler().fit_transform(X_items)
pca = PCA(n_components=5, random_state=SEED)
X_pca = pca.fit_transform(X_std)

loadings = pd.DataFrame(
    pca.components_.T, index=behaviour_item_cols,
    columns=[f'PC{i+1}' for i in range(pca.n_components_)]
)
pc_top_features = {pc: loadings[pc].abs().sort_values(ascending=False).head(5).index.tolist() for pc in loadings.columns}

kmeans = KMeans(n_clusters=3, random_state=SEED, n_init=20)
df['kmeans_cluster'] = kmeans.fit_predict(X_std)

print('PCA explained variance ratio:', np.round(pca.explained_variance_ratio_, 3))
print('Cluster counts:', df['kmeans_cluster'].value_counts().sort_index().to_dict())
for pc, vars_ in pc_top_features.items():
    print(f'{pc} top loadings:', vars_)

PCA explained variance ratio: [0.29  0.086 0.074 0.063 0.058]
Cluster counts: {0: 8, 1: 16, 2: 26}
PC1 top loadings: ['Q26_ConcentrationProblems_num', 'Q33_UseLongerThanIntended_B_num', 'Q30_PhoneOnMyMind_num', 'Q16_UseWhenStressed_num', 'Q22_UnlockWithoutReason_num']
PC2 top loadings: ['Q20_NotificationsWakeNight_num', 'Q8_StopUseAtNight_num', 'Q21_WakeToCheckPhone_num', 'Q7_After11PMUse_num', 'Q19_PhoneDelaysSleep_num']
PC3 top loadings: ['Q10_SocialMediaTime_num', 'Q12_PhoneDuringWorkClass_num', 'Q4_DailyPhoneTime_num', 'Q25_MissPlannedWork_num', 'Q21_WakeToCheckPhone_num']
PC4 top loadings: ['Q29_ImpatientWithoutPhone_num', 'Q6_AfterWakeCheck_num', 'Q28_CannotStandWithoutPhone_num', 'Q5_CheckFrequency_num', 'Q31_WontGiveUpPhone_num']
PC5 top loadings: ['Q27_PhysicalPain_num', 'Q28_CannotStandWithoutPhone_num', 'Q5_CheckFrequency_num', 'Q32_ConstantChecking_num', 'Q11_PhoneDuringMeals_num']


**Step 4 evaluation**
- What the code did: Built interaction features, ran PCA on behavioural items, and clustered users using KMeans.
- What result was obtained: PCA variance ratios are `[0.290, 0.086, 0.074, 0.063, 0.058]`; KMeans cluster counts are `{0: 8, 1: 16, 2: 26}`.
- Why this matters for decisions: These analyses reveal latent structure and profile heterogeneity, helping interpret behaviour patterns alongside Engine 1 outputs.

## Step 5 - Engine 1: behavioural risk tier and top drivers

### Methodology
1. Define the Engine 1 input scope as behavioural-only features.
2. Apply guard checks to ensure no PHQ variables (`Q35` to `Q43`) and no demographic variables are used in the primary trigger.
3. Select the five behavioural domain features as the core risk-building inputs.
4. Estimate domain weights using PCA on domain scores, then normalize these weights so they sum to 1.
5. Compute a per-user `behavioural_risk_index` as a weighted sum of domain scores.
6. Convert the continuous risk index into three intervention tiers (`low`, `moderate`, `high`) using quantile cutoffs.
7. Rank each user’s domain scores and extract top 3 behavioural drivers to explain why they were assigned that tier.
8. Build the Engine 1 output table (`signal_to_decision_df`) with row ID, risk tier, domain profile, top drivers, and PSU-style score.

### What the coding cell is doing in plain language
1. It first protects the model boundary so Engine 1 stays purely behavioural and does not accidentally use mental-health target items or demographics.
2. It then combines domain scores into one overall risk score, but keeps the contribution of each domain visible via weights.
3. It groups people into practical priority bands (low/moderate/high) so intervention decisions are easier to operationalize than using raw scores.
4. It adds human-readable reasons (`top_driver_1..3`) so a stakeholder can understand what behaviour patterns are driving risk.
5. Finally, it packages all outputs into one structured table ready for integration with Engine 2.

### Why this matters in this step
1. This is the main decision engine that determines intervention priority.
2. It balances two needs at once: measurable risk ranking and interpretable behavioural explanation.
3. It creates outputs that can be directly translated into targeted behavioural interventions.

### Why this matters for the full analysis
1. Engine 1 is the primary trigger, so this step establishes the core workflow logic for the project.
2. Strong guardrails here preserve the intended separation between behavioural profiling and PHQ association validation.
3. The resulting structured outputs become the backbone for final reporting, example-user walkthroughs, and operational deployment.

In [5]:
main_behavioural_features = behaviour_item_cols.copy()
assert not any(c.startswith('Q35_') or c.startswith('Q36_') or c.startswith('Q37_') or c.startswith('Q38_') or c.startswith('Q39_') or c.startswith('Q40_') or c.startswith('Q41_') or c.startswith('Q42_') or c.startswith('Q43_') for c in main_behavioural_features), 'PHQ items leaked into Engine 1'
assert not any(c.startswith('Q2_Gender_') or c.startswith('Q3_Status_') or c == 'Q1_Age' for c in main_behavioural_features), 'Demographics leaked into Engine 1'

domain_feature_cols = [
    'domain_exposure_intensity', 'domain_context_disruption', 'domain_emotional_coping',
    'domain_sleep_disruption', 'domain_self_regulation'
]

domain_matrix = df[domain_feature_cols].astype(float)
domain_std = StandardScaler().fit_transform(domain_matrix)
domain_pca = PCA(n_components=1, random_state=SEED).fit(domain_std)
loading_norm = np.abs(domain_pca.components_[0]) / np.abs(domain_pca.components_[0]).sum()
weights = dict(zip(domain_feature_cols, loading_norm))

df['behavioural_risk_index'] = sum(df[k] * v for k, v in weights.items())
q1, q2 = df['behavioural_risk_index'].quantile([0.33, 0.66])
df['behavioural_risk_tier'] = pd.cut(df['behavioural_risk_index'], bins=[-np.inf, q1, q2, np.inf], labels=['low', 'moderate', 'high'])

driver_labels = {
    'domain_exposure_intensity': 'exposure_intensity',
    'domain_context_disruption': 'context_disruption',
    'domain_emotional_coping': 'emotional_coping',
    'domain_sleep_disruption': 'sleep_disruption',
    'domain_self_regulation': 'self_regulation',
}

def top_drivers(row, n=3):
    ranked = sorted(domain_feature_cols, key=lambda c: row[c], reverse=True)[:n]
    return [driver_labels[c] for c in ranked]

drivers = df.apply(top_drivers, axis=1, result_type='expand')
drivers.columns = ['top_driver_1', 'top_driver_2', 'top_driver_3']
df = pd.concat([df, drivers], axis=1)

signal_to_decision_df = pd.DataFrame({
    'row_id': np.arange(len(df)),
    'behavioural_risk_tier': df['behavioural_risk_tier'].astype(str),
    'domain_exposure_intensity': df['domain_exposure_intensity'],
    'domain_context_disruption': df['domain_context_disruption'],
    'domain_emotional_coping': df['domain_emotional_coping'],
    'domain_sleep_disruption': df['domain_sleep_disruption'],
    'domain_self_regulation': df['domain_self_regulation'],
    'top_driver_1': df['top_driver_1'],
    'top_driver_2': df['top_driver_2'],
    'top_driver_3': df['top_driver_3'],
    'psu_style_score': df['psu_style_score'],
})

print('Engine 1 domain weights:', {k: round(float(v), 4) for k, v in weights.items()})
print('Engine 1 risk tier distribution:')
print(signal_to_decision_df['behavioural_risk_tier'].value_counts())
display(signal_to_decision_df.head())

Engine 1 domain weights: {'domain_exposure_intensity': 0.2039, 'domain_context_disruption': 0.1914, 'domain_emotional_coping': 0.2116, 'domain_sleep_disruption': 0.2032, 'domain_self_regulation': 0.1899}
Engine 1 risk tier distribution:
behavioural_risk_tier
low         17
high        17
moderate    16
Name: count, dtype: int64


,row_id,behavioural_risk_tier,domain_exposure_intensity,domain_context_disruption,domain_emotional_coping,domain_sleep_disruption,domain_self_regulation,top_driver_1,top_driver_2,top_driver_3,psu_style_score
0,0,low,37.500000,37.5,58.333333,25.0,41.666667,emotional_coping,self_regulation,exposure_intensity,18.0
1,1,high,91.666667,87.5,100.000000,62.5,100.000000,emotional_coping,self_regulation,exposure_intensity,76.0
2,2,low,45.833333,37.5,58.333333,37.5,58.333333,emotional_coping,self_regulation,exposure_intensity,28.0
3,3,moderate,76.388889,62.5,75.000000,62.5,25.000000,exposure_intensity,emotional_coping,context_disruption,20.0
4,4,high,90.277778,87.5,100.000000,50.0,100.000000,emotional_coping,self_regulation,exposure_intensity,34.0


**Step 5 evaluation**
- What the code did: Constructed Engine 1 risk index, assigned risk tiers, and generated top behavioural drivers with leakage guards.
- What result was obtained: Tier distribution is low `17`, moderate `16`, high `17`; domain weights are near-balanced (about `0.19` to `0.21`).
- Why this matters for decisions: Engine 1 now provides the primary trigger plus interpretable reasons for intervention prioritization.

## Step 6 - Engine 2: PHQ9 association model comparison (secondary only)

### Methodology
1. Build a supervised feature matrix from behavioural variables (plus optional app indicators).
2. Train multiple candidate models with stratified cross-validation.
3. Compare models using metrics relevant to imbalance (ROC-AUC, F1, precision, recall, balanced accuracy).

### Model explanation and observed results (Step 6A)

| Model | Type | What it captures | Observed CV result in this notebook | Interpretation in this notebook |
|---|---|---|---|---|
| `logistic_regression` | Linear classifier with class balancing | Additive behavioural signal | ROC-AUC `0.526`, F1 `0.214`, Precision `0.157`, Recall `0.367`, Balanced Accuracy `0.514` | Best baseline by F1/Recall ranking, but still weak for positive detection |
| `random_forest` | Bagged decision-tree ensemble | Nonlinear interactions | ROC-AUC `0.551`, F1 `0.000`, Recall `0.000`, Balanced Accuracy `0.463` | Failed to recover positives at tested operating behavior |
| `hist_gradient_boosting` | Boosted tree ensemble | Nonlinear residual structure | ROC-AUC `0.626`, F1 `0.000`, Recall `0.000`, Balanced Accuracy `0.500` | Higher ranking by ROC-AUC alone, but not useful for positive-case detection here |

### Why this matters in this step
1. It checks whether behavioural signals are associated with PHQ9 elevation.
2. Model comparison prevents over-reliance on a single algorithm.

### Why this matters for the full analysis
1. Engine 2 adds evidence quality and risk-context information.
2. It remains secondary and supports, rather than replaces, Engine 1.

In [7]:
app_cols = [
    c for c in [
        'Q9_App_SocialMedia','Q9_App_Messaging','Q9_App_ProductivityStudy','Q9_App_WebBrowsing',
        'Q9_App_VideoStreaming','Q9_App_Gaming','Q9_App_Shopping'
    ] if c in df.columns
]
X_sec = df[behaviour_item_cols + app_cols].copy()
for c in X_sec.columns:
    if X_sec[c].dtype == bool:
        X_sec[c] = X_sec[c].astype(int)
X_sec = X_sec.apply(pd.to_numeric, errors='coerce').fillna(X_sec.median(numeric_only=True))
y_sec = df['PHQ9_10plus'].astype(int)

models = {
    'logistic_regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED))
    ]),
    'random_forest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestClassifier(n_estimators=400, class_weight='balanced', random_state=SEED))
    ]),
    'hist_gradient_boosting': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', HistGradientBoostingClassifier(random_state=SEED))
    ])
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {
    'roc_auc': 'roc_auc',
    'f1': 'f1',
    'precision': 'precision',
    'recall': 'recall',
    'balanced_accuracy': 'balanced_accuracy'
}

rows = []
for name, model in models.items():
    cv_res = cross_validate(model, X_sec, y_sec, cv=cv, scoring=scoring, n_jobs=-1, error_score='raise')
    rows.append({
        'model': name,
        'roc_auc_mean': np.mean(cv_res['test_roc_auc']),
        'f1_mean': np.mean(cv_res['test_f1']),
        'precision_mean': np.mean(cv_res['test_precision']),
        'recall_mean': np.mean(cv_res['test_recall']),
        'balanced_accuracy_mean': np.mean(cv_res['test_balanced_accuracy'])
    })

comparison_df = pd.DataFrame(rows).sort_values(['f1_mean', 'recall_mean', 'roc_auc_mean'], ascending=False).reset_index(drop=True)
best_model_name = comparison_df.loc[0, 'model']
best_model = models[best_model_name]
print('Selected Engine 2 model:', best_model_name)
display(comparison_df)

Selected Engine 2 model: logistic_regression


,model,roc_auc_mean,f1_mean,precision_mean,recall_mean,balanced_accuracy_mean
0,logistic_regression,0.525595,0.214286,0.156667,0.366667,0.51369
1,hist_gradient_boosting,0.626190,0.000000,0.000000,0.000000,0.50000
2,random_forest,0.550595,0.000000,0.000000,0.000000,0.46250


**Step 6 evaluation**
- What the code did: Trained baseline Engine 2 models on full behavioural features with stratified cross-validation and compared metrics.
- What result was obtained: Baseline winner is logistic regression with ROC-AUC `0.526`, F1 `0.214`, Precision `0.157`, Recall `0.367`, Balanced Accuracy `0.514`.
- Why this matters for decisions: Baseline positive-case detection is weak, so model refinement is needed before using Engine 2 for contextual prioritization.

## Step 6B - Improvement experiment: simpler and regularized secondary models

### Methodology
1. Compare full-feature logistic model against lower-dimensional domain-only variants.
2. Add stronger regularization to reduce overfitting risk in small-sample settings.
3. Re-select the best secondary model if improved metrics are observed.

### Model explanation and observed results (Step 6B)

| Model | Feature set | Regularization | Observed CV result in this notebook | Interpretation in this notebook |
|---|---|---|---|---|
| `logreg_full_l2` | Full behavioural items + app indicators | L2 (`C=1.0`) | ROC-AUC `0.526`, AP `0.351`, F1 `0.214`, Recall `0.367`, Balanced Accuracy `0.514` | Baseline reference model; weaker detection on this sample |
| `logreg_domain_l2` | Domain scores only | L2 (`C=1.0`) | ROC-AUC `0.679`, AP `0.536`, F1 `0.471`, Recall `0.700`, Balanced Accuracy `0.664` | Best overall trade-off; selected final Engine 2 model |
| `logreg_domain_strong_reg` | Domain scores only | Stronger L2 (`C=0.25`) | ROC-AUC `0.698`, AP `0.554`, F1 `0.441`, Recall `0.600`, Balanced Accuracy `0.643` | Strong discrimination, but lower F1/Recall than selected model |

### Why this matters in this step
1. Simpler models can be more stable when observations are limited.
2. Regularization improves generalization by limiting noise-fitting.

### Why this matters for the full analysis
1. Better Engine 2 stability makes supporting evidence more reliable.
2. Domain-only models also keep secondary predictions easier to explain.

In [11]:
X_sec_domain = df[domain_feature_cols].copy()
y_sec_domain = y_sec.copy()

improve_models = {
    'logreg_full_l2': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED, C=1.0, penalty='l2'))
    ]),
    'logreg_domain_l2': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED, C=1.0, penalty='l2'))
    ]),
    'logreg_domain_strong_reg': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED, C=0.25, penalty='l2'))
    ])
}

improve_rows = []
for mname, mobj in improve_models.items():
    X_use = X_sec_domain if 'domain' in mname else X_sec
    res = cross_validate(
        mobj, X_use, y_sec_domain, cv=cv, n_jobs=-1, scoring={
            'roc_auc': 'roc_auc',
            'f1': 'f1',
            'precision': 'precision',
            'recall': 'recall',
            'balanced_accuracy': 'balanced_accuracy',
            'ap': 'average_precision'
        }, error_score='raise'
    )
    improve_rows.append({
        'model': mname,
        'roc_auc_mean': np.mean(res['test_roc_auc']),
        'ap_mean': np.mean(res['test_ap']),
        'f1_mean': np.mean(res['test_f1']),
        'precision_mean': np.mean(res['test_precision']),
        'recall_mean': np.mean(res['test_recall']),
        'balanced_accuracy_mean': np.mean(res['test_balanced_accuracy'])
    })

improve_df = pd.DataFrame(improve_rows).sort_values(
    ['f1_mean', 'recall_mean', 'roc_auc_mean'], ascending=False
).reset_index(drop=True)
display(improve_df)

if improve_df.loc[0, 'model'] != 'logreg_full_l2':
    chosen = improve_df.loc[0, 'model']
    best_model_name = chosen
    best_model = improve_models[chosen]
    use_domain_only = 'domain' in chosen
else:
    use_domain_only = False

print('Improvement-selected Engine 2 model:', best_model_name)
print('Use domain-only features for final fit:', use_domain_only)

,model,roc_auc_mean,ap_mean,f1_mean,precision_mean,recall_mean,balanced_accuracy_mean
0,logreg_domain_l2,0.679167,0.536349,0.471429,0.360000,0.700000,0.664286
1,logreg_domain_strong_reg,0.698214,0.554127,0.440952,0.376667,0.600000,0.642857
2,logreg_full_l2,0.525595,0.350635,0.214286,0.156667,0.366667,0.513690


Improvement-selected Engine 2 model: logreg_domain_l2
Use domain-only features for final fit: True


**Step 6B evaluation**
- What the code did: Compared simplified and regularized logistic variants, including domain-only feature sets, and reselected Engine 2.
- What result was obtained: Best model is `logreg_domain_l2` with ROC-AUC `0.679`, AP `0.536`, F1 `0.471`, Precision `0.360`, Recall `0.700`, Balanced Accuracy `0.664`.
- Why this matters for decisions: A lower-dimensional, interpretable feature space substantially improves Engine 2 reliability for supporting risk context.

### Conclusion after Step 6A and Step 6B
1. Step 6A baseline winner was logistic regression on full behavioural features, but performance was limited (F1 `0.214`, Recall `0.367`).
2. Step 6B refinement showed that domain-only logistic (`logreg_domain_l2`) performs clearly better on detection-oriented metrics.
3. Final best Engine 2 model for this notebook is **`logreg_domain_l2` (domain-score logistic regression)**.
4. This model should be used for downstream threshold analysis (Step 7), uncertainty analysis (Step 7B), and final integration (Step 8).

## Step 7 - Threshold and confusion-matrix trade-off

### Methodology
1. Use holdout probabilities from the selected Engine 2 model.
2. Evaluate multiple probability thresholds and compute precision, recall, F1, and balanced accuracy.
3. Select an operating threshold and inspect confusion matrix errors (FP/FN/TP/TN).

### Why this matters in this step
1. Threshold choice determines practical model behavior, not just model quality.
2. Confusion-matrix trade-offs make false-alarm vs missed-risk costs explicit.

### Why this matters for the full analysis
1. It turns model scores into actionable policy settings.
2. It supports safer deployment by aligning threshold choice with intervention objectives.

In [12]:
X_work = X_sec_domain if ('use_domain_only' in globals() and use_domain_only) else X_sec

X_train, X_test, y_train, y_test = train_test_split(
    X_work, y_sec, test_size=0.25, stratify=y_sec, random_state=SEED
)
best_model.fit(X_train, y_train)
test_proba = best_model.predict_proba(X_test)[:, 1]

thresholds = [0.30, 0.40, 0.50]
thr_rows = []
for thr in thresholds:
    pred = (test_proba >= thr).astype(int)
    thr_rows.append({
        'threshold': thr,
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y_test, pred)
    })
thr_df = pd.DataFrame(thr_rows)

grid = np.round(np.arange(0.05, 0.96, 0.05), 2)
grid_rows = []
for thr in grid:
    pred = (test_proba >= thr).astype(int)
    grid_rows.append({
        'threshold': thr,
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y_test, pred)
    })
grid_df = pd.DataFrame(grid_rows)
best_f1_row = grid_df.sort_values(['f1', 'recall', 'balanced_accuracy'], ascending=False).iloc[0]
best_bal_row = grid_df.sort_values(['balanced_accuracy', 'recall'], ascending=False).iloc[0]

chosen_threshold = float(best_f1_row['threshold'])
test_pred = (test_proba >= chosen_threshold).astype(int)
cm = confusion_matrix(y_test, test_pred)
tn, fp, fn, tp = cm.ravel()

print('Engine 2 model for threshold analysis:', best_model_name)
display(thr_df)
print('Top 5 thresholds by F1/recall:')
display(grid_df.sort_values(['f1', 'recall', 'balanced_accuracy'], ascending=False).head(5))
print(f"Recommended threshold by F1: {best_f1_row['threshold']:.2f}")
print(f"Alternative threshold by balanced accuracy: {best_bal_row['threshold']:.2f}")
print('Confusion matrix at recommended threshold [[TN, FP],[FN, TP]]:\n', cm)
print(f'FP={fp}, FN={fn}, TP={tp}, TN={tn}')

Engine 2 model for threshold analysis: logreg_domain_l2


,threshold,precision,recall,f1,balanced_accuracy
0,0.3,0.285714,0.666667,0.400000,0.583333
1,0.4,0.285714,0.666667,0.400000,0.583333
2,0.5,0.333333,0.666667,0.444444,0.633333


Top 5 thresholds by F1/recall:


,threshold,precision,recall,f1,balanced_accuracy
9,0.50,0.333333,0.666667,0.444444,0.633333
3,0.20,0.285714,0.666667,0.400000,0.583333
4,0.25,0.285714,0.666667,0.400000,0.583333
5,0.30,0.285714,0.666667,0.400000,0.583333
6,0.35,0.285714,0.666667,0.400000,0.583333


Recommended threshold by F1: 0.50
Alternative threshold by balanced accuracy: 0.50
Confusion matrix at recommended threshold [[TN, FP],[FN, TP]]:
 [[6 4]
 [1 2]]
FP=4, FN=1, TP=2, TN=6


**Step 7 evaluation**
- What the code did: Performed threshold sweep, selected a recommended threshold, and computed confusion-matrix trade-offs.
- What result was obtained: Recommended threshold is `0.50`; at this threshold Precision `0.333`, Recall `0.667`, F1 `0.444`, Balanced Accuracy `0.633`, confusion matrix `[[6, 4], [1, 2]]`.
- Why this matters for decisions: This operating point reduces false negatives (`1`) while keeping detection useful, improving safety for intervention triage.

## Step 7B - Bootstrap confidence intervals for uncertainty

### Methodology
1. Re-sample holdout predictions many times using bootstrap sampling.
2. Recompute F1, recall, and balanced accuracy for each bootstrap sample.
3. Estimate uncertainty ranges (for example 95% intervals) for each metric.

### Why this matters in this step
1. It quantifies how stable the holdout metrics are under sampling variation.
2. It avoids overconfidence from one split and one point estimate.

### Why this matters for the full analysis
1. Uncertainty-aware reporting makes secondary evidence safer to interpret.
2. It reinforces the policy that Engine 2 is supporting context, not a standalone trigger.

In [15]:
rng = np.random.default_rng(SEED)
B = 2000

boot_rows = []
for _ in range(B):
    idx = rng.integers(0, len(y_test), len(y_test))
    y_b = y_test.to_numpy()[idx]
    p_b = test_proba[idx]
    pred_b = (p_b >= chosen_threshold).astype(int)
    boot_rows.append({
        'f1': f1_score(y_b, pred_b, zero_division=0),
        'recall': recall_score(y_b, pred_b, zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y_b, pred_b)
    })

boot_df = pd.DataFrame(boot_rows)
ci_df = pd.DataFrame({
    'metric': ['f1', 'recall', 'balanced_accuracy'],
    'mean': [boot_df['f1'].mean(), boot_df['recall'].mean(), boot_df['balanced_accuracy'].mean()],
    'ci_2_5': [boot_df['f1'].quantile(0.025), boot_df['recall'].quantile(0.025), boot_df['balanced_accuracy'].quantile(0.025)],
    'ci_97_5': [boot_df['f1'].quantile(0.975), boot_df['recall'].quantile(0.975), boot_df['balanced_accuracy'].quantile(0.975)],
})

print('Bootstrap runs:', B)
print('Chosen threshold used for bootstrap:', chosen_threshold)
display(ci_df)

Bootstrap runs: 2000
Chosen threshold used for bootstrap: 0.5


,metric,mean,ci_2_5,ci_97_5
0,f1,0.410057,0.00,0.800000
1,recall,0.640452,0.00,1.000000
2,balanced_accuracy,0.629427,0.25,0.909091


**Step 7B evaluation**
- What the code did: Used bootstrap resampling (`2000` runs) at threshold `0.50` to estimate uncertainty intervals for holdout metrics.
- What result was obtained: F1 mean `0.410` (95% CI `0.000` to `0.800`), Recall mean `0.640` (95% CI `0.000` to `1.000`), Balanced Accuracy mean `0.629` (95% CI `0.250` to `0.909`).
- Why this matters for decisions: Wide intervals indicate high uncertainty in small samples, reinforcing that Engine 2 should remain supporting evidence.

## Step 8 - Final signal-to-decision table

### Methodology
1. Refit the selected Engine 2 model on the chosen final feature matrix.
2. Generate per-user PHQ association probabilities.
3. Merge Engine 2 probabilities with Engine 1 outputs into one final schema.

### Why this matters in this step
1. It creates the final table used for operational review and reporting.
2. It preserves one-row-per-user traceability for decision support.

### Why this matters for the full analysis
1. It integrates both engines without violating role boundaries.
2. It enables downstream teams to act on a single, consistent artifact.

In [13]:
X_final = X_sec_domain if ('use_domain_only' in globals() and use_domain_only) else X_sec
best_model.fit(X_final, y_sec)
signal_to_decision_df['phq9_association_risk_estimate'] = best_model.predict_proba(X_final)[:, 1]

final_cols = [
    'row_id', 'behavioural_risk_tier',
    'domain_exposure_intensity', 'domain_context_disruption', 'domain_emotional_coping',
    'domain_sleep_disruption', 'domain_self_regulation',
    'top_driver_1', 'top_driver_2', 'top_driver_3',
    'psu_style_score', 'phq9_association_risk_estimate'
]
signal_to_decision_df = signal_to_decision_df[final_cols]

display(signal_to_decision_df.head(10))

,row_id,behavioural_risk_tier,domain_exposure_intensity,domain_context_disruption,domain_emotional_coping,domain_sleep_disruption,domain_self_regulation,top_driver_1,top_driver_2,top_driver_3,psu_style_score,phq9_association_risk_estimate
0,0,low,37.500000,37.50,58.333333,25.00,41.666667,emotional_coping,self_regulation,exposure_intensity,18.0,0.131240
1,1,high,91.666667,87.50,100.000000,62.50,100.000000,emotional_coping,self_regulation,exposure_intensity,76.0,0.807329
2,2,low,45.833333,37.50,58.333333,37.50,58.333333,emotional_coping,self_regulation,exposure_intensity,28.0,0.119221
3,3,moderate,76.388889,62.50,75.000000,62.50,25.000000,exposure_intensity,emotional_coping,context_disruption,20.0,0.538112
4,4,high,90.277778,87.50,100.000000,50.00,100.000000,emotional_coping,self_regulation,exposure_intensity,34.0,0.826863
5,5,low,75.000000,62.50,33.333333,50.00,50.000000,exposure_intensity,context_disruption,sleep_disruption,44.0,0.214152
6,6,low,54.166667,43.75,75.000000,31.25,50.000000,emotional_coping,exposure_intensity,self_regulation,16.0,0.294397
7,7,low,52.777778,87.50,41.666667,31.25,16.666667,context_disruption,exposure_intensity,emotional_coping,14.0,0.441292
8,8,low,63.888889,43.75,75.000000,43.75,33.333333,emotional_coping,exposure_intensity,context_disruption,22.0,0.350187
9,9,low,47.222222,50.00,50.000000,31.25,58.333333,self_regulation,context_disruption,emotional_coping,18.0,0.149113


**Step 8 evaluation**
- What the code did: Refit the selected Engine 2 model on final features and appended `phq9_association_risk_estimate` to the Engine 1 output table.
- What result was obtained: Final table now contains both primary behavioural signals and secondary PHQ association probability per user.
- Why this matters for decisions: A single integrated output supports consistent, operational review without mixing primary and secondary roles.

## Step 9 - Example users (3 contrasting users)

### Methodology
1. Select three contrasting users from the final output (typically low, moderate, high behavioural tiers).
2. Display each user’s Engine 1 profile (tier, domain scores, top drivers) and Engine 2 probability.
3. Compare alignment and mismatch patterns across the three users.

### Why this matters in this step
1. It demonstrates concrete end-to-end behaviour of both engines on real rows.
2. It translates model outputs into case-level interpretation for non-technical readers.

### Why this matters for the full analysis
1. It proves how primary trigger logic and secondary context work together.
2. It strengthens communication quality for intervention stakeholders and reviewers.

In [14]:
tier_sort = {'low': 0, 'moderate': 1, 'high': 2}
view_df = signal_to_decision_df.copy()
view_df['tier_ord'] = view_df['behavioural_risk_tier'].map(tier_sort)
view_df = view_df.sort_values(['tier_ord', 'phq9_association_risk_estimate']).reset_index(drop=True)

picked_ids = []
for tier in ['low', 'moderate', 'high']:
    subset = signal_to_decision_df[signal_to_decision_df['behavioural_risk_tier'] == tier]
    if len(subset) > 0:
        picked_ids.append(int(subset.sort_values('phq9_association_risk_estimate', ascending=False).iloc[0]['row_id']))

example_users_df = signal_to_decision_df[signal_to_decision_df['row_id'].isin(picked_ids)].sort_values('behavioural_risk_tier')
display(example_users_df)

for _, r in example_users_df.iterrows():
    print(f"row_id={int(r['row_id'])} | tier={r['behavioural_risk_tier']} | PHQ-assoc={r['phq9_association_risk_estimate']:.3f} | drivers={r['top_driver_1']}, {r['top_driver_2']}, {r['top_driver_3']}")

,row_id,behavioural_risk_tier,domain_exposure_intensity,domain_context_disruption,domain_emotional_coping,domain_sleep_disruption,domain_self_regulation,top_driver_1,top_driver_2,top_driver_3,psu_style_score,phq9_association_risk_estimate
29,29,high,91.666667,87.5,91.666667,37.50,66.666667,exposure_intensity,emotional_coping,context_disruption,48.0,0.861725
7,7,low,52.777778,87.5,41.666667,31.25,16.666667,context_disruption,exposure_intensity,emotional_coping,14.0,0.441292
38,38,moderate,72.222222,75.0,83.333333,50.00,41.666667,emotional_coping,context_disruption,exposure_intensity,34.0,0.671996


row_id=29 | tier=high | PHQ-assoc=0.862 | drivers=exposure_intensity, emotional_coping, context_disruption
row_id=7 | tier=low | PHQ-assoc=0.441 | drivers=context_disruption, exposure_intensity, emotional_coping
row_id=38 | tier=moderate | PHQ-assoc=0.672 | drivers=emotional_coping, context_disruption, exposure_intensity


**Step 9 evaluation and interpretation**
- What the code did: Selected three contrasting users (low, moderate, high tiers) and displayed tier, drivers, and PHQ association probability together.
- What result was obtained: Current examples are row `29` (high, `0.862`), row `38` (moderate, `0.672`), and row `7` (low, `0.441`).
- Why this matters for decisions: Case-level traces show how Engine 1 explains behavioural need and Engine 2 adds context for urgency calibration.

## Final operational policy
- Primary trigger: `behavioural_risk_tier` from Engine 1.
- Supporting evidence: `phq9_association_risk_estimate` from Engine 2.
- Intervention decisions remain behaviour-focused and interpretable; Engine 2 does not replace Engine 1.